# Chặng 6 - Thực nghiệm: Hệ thống Phân loại Cảm xúc tiếng Việt (Sentiment Analysis)

**Trường Đại học Nam Cần Thơ - Môn: Máy học nâng cao (25MIT-1A)**
**Giảng viên:** TS. Huỳnh Văn Huy
**Nhóm:** Võ Khương Duy (2513464) - Nguyễn Thị Mai Hân (2513562) - Nguyễn Minh Nhựt (2513525)

## Quy trình tổng quát (9 bước)

```
UIT-VSFC (3 lớp) -> Tiền xử lý -> Tokenizer PhoBERT -> Train/Valid/Test
   -> (1) TF-IDF + Logistic Regression (baseline)
   -> (2) PhoBERT fine-tuned sẵn (đối chứng, không huấn luyện)
   -> (3) Fine-tuning PhoBERT-base-v2 (GPU Colab T4)
   -> Đánh giá (Accuracy/P/R/F1/CM/PR curve) -> Lưu model local
   -> Flask web (Tailwind) + Cloudflared tunnel -> demo public
```

Notebook này **clone mã nguồn từ GitHub** (hoặc đọc từ Google Drive) rồi chạy toàn bộ thực nghiệm trên Colab - không cần máy mạnh cục bộ. Cuối notebook có **web demo** (Flask + Tailwind) expose qua tunnel để trình bày, thử nghiệm trực quan.

## 1. Thiết lập môi trường

In [ ]:
# Kiểm tra GPU (Colab free: Tesla T4)
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cài đặt thư viện
!pip install -q transformers scikit-learn matplotlib seaborn pandas numpy huggingface_hub


In [ ]:
# (Tuỳ chọn) Xác thực Hugging Face bằng Colab secrets - chỉ cần nếu model gated
# Cách đặt: biểu tượng khoá bên trái Colab -> "+ New secret" -> tên HF_TOKEN
# Không cần chạy nếu model public (phobert-base-v2, wonrax đều public)
try:
    from google.colab import userdata
    from huggingface_hub import login
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Da dang nhap Hugging Face (HF_TOKEN)")
    else:
        print("Chua co HF_TOKEN - bo qua (model public, khong can token)")
except Exception:
    print("Chua cau hinh Colab secrets - bo qua (model public, khong can token)")


## 2. Lấy mã nguồn

Clone repo `https://github.com/duyvo26/code_giua_ki_train_AI`. Nếu chưa push GitHub, chạy cell bên dưới để **mount Google Drive** và để thư mục `code_giua_ki_train_AI` trong MyDrive.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/duyvo26/code_giua_ki_train_AI"
PROJECT_DIR = "code_giua_ki_train_AI"

if Path(PROJECT_DIR).is_dir():
    print("-> Đã có mã nguồn trong thư mục hiện tại.")
elif Path(f"/content/drive/MyDrive/{PROJECT_DIR}").is_dir():
    PROJECT_DIR = f"/content/drive/MyDrive/{PROJECT_DIR}"
    print("-> Tìm thấy mã nguồn trên Google Drive.")
else:
    print("-> Chưa có mã nguồn, thử git clone...")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print("git clone THẤT BẠI. Chạy cell bên dưới để mount Google Drive")
        print("rồi tải thư mục code_giua_ki_train_AI vào MyDrive, sau đó chạy lại cell này.")
        print(result.stderr[-2000:])
        raise SystemExit(1)

PROJECT_DIR = os.path.abspath(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print("-> Thư mục làm việc:", PROJECT_DIR)


In [ ]:
# Chỉ chạy khi chưa có GitHub - mount Google Drive để lấy mã nguồn
from google.colab import drive
drive.mount("/content/drive")


## 3. Dữ liệu & Tiền xử lý (Bước 1-3)

Bộ dữ liệu: **UIT-VSFC** (Vietnamese Students' Feedback Corpus, paper KSE 2018) - ~11.000 bình luận tiếng Việt, 3 nhãn:
- `0` = Negative (Tiêu cực) - `1` = Neutral (Trung tính) - `2` = Positive (Tích cực)

Dữ liệu được tải từ Hugging Face Hub với split **train/valid/test chính thức theo paper (ti lệ ~80/10/10)**. Test set hoàn toàn tách biệt, không dùng trong fine-tuning.

In [ ]:
from scripts.preprocess import prepare_dataset, show_summary

splits, summaries = prepare_dataset()
show_summary(summaries)


In [ ]:
# Trực quan phân bố nhãn (kiểm tra mất cân bằng)
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from scripts.config import LABEL_NAMES_VI

Path("results/figures").mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, df) in zip(axes, splits.items()):
    counts = df["label_id"].value_counts().sort_index()
    ax.bar(LABEL_NAMES_VI, counts, color=["#d62728", "#ffbf00", "#2ca02c"])
    ax.set_title(name.upper())
    ax.set_ylabel("Số mẫu")
    for i, v in enumerate(counts):
        ax.text(i, v + 20, str(v), ha="center")
plt.tight_layout()
plt.savefig("results/figures/01_label_distribution.png", dpi=150)
plt.show()


## 4. Thực nghiệm 1 - Baseline: TF-IDF + Logistic Regression

Mô hình truyền thống làm **đối chứng** để chứng minh lợi thế của Transformer.

In [ ]:
from scripts.baseline import run_baseline

baseline_metrics = run_baseline(splits)


## 5. Thực nghiệm 2 - PhoBERT fine-tuned sẵn (đối chứng, không huấn luyện)

Đánh giá trực tiếp mô hình công khai `wonrax/phobert-base-vietnamese-sentiment` (PhoBERT được fine-tune sẵn trên tiếng Việt) trên tập test.

> **Ghi chú khoa học:** mô hình này được fine-tune sẵn (không phải của nhóm) nên điểm số chỉ mang tính **tham chiếu** - cho biết mức năng lực của PhoBERT với bài toán này. Mô hình fine-tuned của nhóm (Thực nghiệm 3) được huấn luyện sạch trên split train chính thức nên kết quả là con số trung thực cho báo cáo.
>
> **Lưu ý kỹ thuật:** thứ tự nhãn của model sẵn có có thể khác chuẩn của dự án (vd {0:'NEG', 1:'POS', 2:'NEU'}) nên hàm `evaluate_transformer` tự **align nhãn** về 0=Negative, 1=Neutral, 2=Positive trước khi tính metrics.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from scripts.evaluate import evaluate_transformer, print_metrics_table, save_confusion_matrix, save_pr_curve, save_metrics_json
from scripts.config import PUBLIC_SENTIMENT_MODEL

model_name_public = PUBLIC_SENTIMENT_MODEL
print("Model doi chung:", model_name_public)

tokenizer = AutoTokenizer.from_pretrained(model_name_public)
model = AutoModelForSequenceClassification.from_pretrained(model_name_public)

id2label = getattr(model.config, "id2label", {})
print("id2label:", id2label)

pub_metrics, y_true, y_pred, proba = evaluate_transformer(
    model, tokenizer, splits["test"],
    model_name="PhoBERT fine-tuned sẵn (tham chiếu)",
    id2label=id2label,
)
print_metrics_table(pub_metrics)
save_confusion_matrix(y_true, y_pred, "public_pretrained", "Confusion Matrix - PhoBERT fine-tuned sẵn")
save_pr_curve(y_true, proba, "public_pretrained", "Precision-Recall - PhoBERT fine-tuned sẵn")
save_metrics_json(pub_metrics, "public_pretrained")


## 6. Thực nghiệm 3 - Fine-tuning PhoBERT-base-v2 (điểm chính của bài)

- Tải mô hình mã nguồn mở `vinai/phobert-base-v2` (bản cập nhật, pretrained trên tiếng Việt).
- Gắn classification head 3 lớp, tối ưu **Cross-Entropy** bằng Trainer.
- Chạy trên **GPU T4** của Colab với FP16 (thời gian dự kiến ~15-20 phút cho 3 epochs).

In [ ]:
from scripts.finetune import fine_tune

trainer = fine_tune(splits)


## 7. Đánh giá mô hình fine-tuned trên tập test (Bước 7)

Tập test chưa từng xuất hiện trong quá trình huấn luyện → kết quả là khả năng tổng quát thực sự.

In [ ]:
from scripts.finetune import load_sentiment_model
from scripts.evaluate import evaluate_transformer, print_metrics_table, save_confusion_matrix, save_pr_curve, save_metrics_json

model, tokenizer = load_sentiment_model()
ft_metrics, y_true, y_pred, proba = evaluate_transformer(
    model, tokenizer, splits["test"], model_name="PhoBERT-base-v2 (fine-tuned)",
)
print_metrics_table(ft_metrics)
save_confusion_matrix(y_true, y_pred, "phobert_finetuned", "Confusion Matrix - PhoBERT fine-tuned")
save_pr_curve(y_true, proba, "phobert_finetuned", "Precision-Recall - PhoBERT fine-tuned")
save_metrics_json(ft_metrics, "phobert_finetuned")


## 8. Bảng so sánh 3 mô hình (trích vào báo cáo)

In [ ]:
from scripts.evaluate import compare_models

compare_table_path = compare_models([baseline_metrics, pub_metrics, ft_metrics])


## 9. Demo Inference - dự báo bình luận mới (Bước 9)

Câu ví dụ của đề bài: **"Sản phẩm rất tệ!"** - model sẽ xuất xác suất % cho 3 lớp.

In [ ]:
from scripts.finetune import predict_sentiment

examples = [
    "Sản phẩm rất tệ!",
    "Sản phẩm rất tốt, giao hàng nhanh",
    "Chất lượng tạm được",
    "Giảng viên nhiệt tình, giảng bài dễ hiểu",
    "Sản phẩm tệ, dùng 2 ngày đã hỏng, không đáng tiền",
]

for text in examples:
    result = predict_sentiment(text, model, tokenizer)
    probs = " | ".join(f"{k}: {v*100:.1f}%" for k, v in result["probabilities"].items())
    print(f"Câu: {text}")
    print(f"  -> {result['sentiment']} ({result['sentiment_vi']}) | độ tin cậy: {result['confidence']*100:.1f}%")
    print(f"     {probs}\n")


## 10. Lưu kết quả (tuỳ chọn - copy sang Google Drive)

Lưu toàn bộ kết quả (metrics JSON, hình ảnh, bảng so sánh) và mô hình tốt nhất sang Google Drive để phục vụ báo cáo và demo web Flask.

In [ ]:
# Chỉ chạy nếu muốn lưu sang Drive (đã mount ở bước 2)
from google.colab import drive
import shutil, os

try:
    drive.mount("/content/drive")
    out_dir = "/content/drive/MyDrive/code_giua_ki_train_AI"
    shutil.copytree("results", os.path.join(out_dir, "results"), dirs_exist_ok=True)
    shutil.copytree("models", os.path.join(out_dir, "models"), dirs_exist_ok=True)
    print("Đã lưu kết quả + mô hình vào Google Drive:", out_dir)
except Exception as exc:
    print("Bỏ qua (không mount Drive):", exc)


## 11. Web demo - Flask + Cloudflared tunnel

Chạy **Flask web** (giao diện Tailwind) trong Colab và expose ra link public bằng Cloudflared:
- **Xem thông tin model** đã train (accuracy, F1, recall Negative).
- **Train lại** mô hình từ web (chạy nền, không treo trang).
- **Dự đoán cảm xúc** bình luận mới với xác suất % 3 lớp.

> Cell dưới chạy Flask trong thread nền rồi in link public để mở trên trình duyệt (điện thoại/máy tính đều truy cập được).

In [ ]:
# Cài thư viện cho web + tunnel
!pip install -q flask requests cloudflared


In [ ]:
# Chạy Flask web trong thread nền (port 8080)
import threading
from webapp.app import app as flask_app

threading.Thread(
    target=flask_app.run,
    kwargs={"host": "0.0.0.0", "port": 8080, "use_reloader": False},
    daemon=True,
).start()
print("Flask da chay tren http://localhost:8080")


In [ ]:
# Mở tunnel Cloudflared -> in link public
import subprocess
import re
import time
import requests

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
timeout = 90
start = time.time()
for line in proc.stdout:
    print(line, end="")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    if time.time() - start > timeout:
        print("HET THOI GIAN cho: khong lay duoc link tunnel")
        break

print("\n======================================================")
print("LINK PUBLIC - mo trong trinh duyet:", public_url)
print("======================================================")


## Kết luận & việc cần làm tiếp theo

1. Điền kết quả từ `results/compare_table.md` vào bảng so sánh trong báo cáo.
2. Phân tích confusion matrix + PR curve, đặc biệt **Recall lớp Negative** (bao nhiêu phàn nàn bị bỏ sót).
3. Dùng **link public của web demo** (cell 11) để chụp ảnh minh hoạ: xem thông tin model, dự đoán `"Sản phẩm rất tệ!"`.
4. Đưa `models/best_model/` về máy/server nội bộ chạy demo (chưa cần tunnel):
   ```bash
   cd code_giua_ki_train_AI
   pip install -r webapp/requirements.txt
   python webapp/app.py
   # mở http://localhost:8080
   ```
5. Kiểm chứng bảo mật: dữ liệu khách hàng không rời khỏi server - mọi dự đoán chạy cục bộ.

> **Lưu ý khoa học:** không cam kết độ chính xác >95% như đề bài gợi ý. Báo cáo số liệu THỰC tế đo được trên tập test và diễn giải (Accuracy, F1, Recall lớp Negative).